# 🎼 Matchmaker 예시 — score following으로 웹 반응 만들기

이 노트북은 **Matchmaker 세션의 수업용 프로토타입**입니다.

핵심 질문은 간단합니다.

> 피아노 연주가 지금 악보의 어디에 있는지 알 수 있다면,
> 웹 비주얼이나 무대 장치를 **구조적으로 반응**하게 만들 수 있지 않을까?

이 노트북에서는 실제 공연용 전체 시스템을 바로 구현하지 않고,
다음 세 단계를 **예시로** 보여줍니다.

1. MIDI 악보에서 **구조 정보 추출**
2. "라이브 연주"를 단순 시뮬레이션하여 **score following 개념 이해**
3. 웹에서 쓸 수 있는 **구간 이벤트 JSON** 만들기

**도구**: `pretty_midi`, `mido`, `matplotlib`, `numpy`

▶ 먼저 아래 셀을 실행해주세요. 설치에 1~2분 정도 걸립니다.


In [ ]:
!pip install -q pretty_midi mido matplotlib numpy

import json
import os
import warnings
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import pretty_midi
from IPython.display import Audio, display

warnings.filterwarnings('ignore')
print('✅ 설치 완료!')


---
## 1. MIDI 파일 올리기

▶ score following의 기준이 될 MIDI 파일을 업로드하세요.

MIDI가 없으면 이 셀은 건너뛰고, 다음 셀에서 예시 MIDI를 사용하세요.


In [ ]:
try:
    from google.colab import files
    uploaded = files.upload()
except Exception:
    uploaded = {}

if uploaded:
    input_midi = list(uploaded.keys())[0]
    print(f'✅ 업로드 완료: {input_midi}')
else:
    print('⚠️ 업로드된 파일이 없습니다. 다음 셀에서 예시 MIDI를 사용하세요.')


---
## 2. (선택) 예시 MIDI 사용

▶ 직접 준비한 MIDI가 없으면 이 셀을 실행하세요.

수업용 예시 멜로디를 다운로드합니다.


In [ ]:
import urllib.request

REPO_URL = 'https://raw.githubusercontent.com/joonhyungbae/pianokit/main/assets'
input_midi = 'melody_example.mid'

if not os.path.exists(input_midi):
    urllib.request.urlretrieve(f'{REPO_URL}/{input_midi}', input_midi)
    print(f'✅ 다운로드 완료: {input_midi}')
else:
    print(f'✅ 이미 존재합니다: {input_midi}')


---
## 3. 악보에서 구조 정보 뽑기

Matchmaker 같은 score following 시스템은 단순히 소리의 크기만 보는 것이 아니라,
**악보 구조 안에서 현재 어디를 연주 중인지** 추적합니다.

먼저 MIDI 악보에서 음표 시작 시점과 음높이를 추출해봅니다.


In [ ]:
pm = pretty_midi.PrettyMIDI(input_midi)
notes = sorted(
    [note for instrument in pm.instruments for note in instrument.notes],
    key=lambda note: note.start,
)

score_onsets = np.array([note.start for note in notes], dtype=float)
score_pitches = np.array([note.pitch for note in notes], dtype=int)
score_durations = np.array([note.end - note.start for note in notes], dtype=float)

print(f'🎹 음표 수: {len(notes)}')
print(f'⏱️ 전체 길이: {pm.get_end_time():.2f}초')
print('첫 8개 음표 예시:')
for idx, note in enumerate(notes[:8]):
    print(f'  {idx+1:02d}. start={note.start:.2f}s pitch={note.pitch} dur={note.end - note.start:.2f}s')

plt.figure(figsize=(12, 4))
plt.scatter(score_onsets, score_pitches, s=60, c=np.linspace(0, 1, len(score_onsets)), cmap='viridis')
plt.title('MIDI score에서 추출한 음표 시작 시점')
plt.xlabel('time (s)')
plt.ylabel('MIDI pitch')
plt.grid(alpha=0.25)
plt.show()


---
## 4. "라이브 연주" 시뮬레이션

실제 연주는 메트로놈처럼 완벽히 일정하지 않습니다.
루바토도 있고, 구간마다 미세하게 빨라지거나 느려지기도 합니다.

아래 셀은 score following 개념을 설명하기 위해,
MIDI 악보의 시작 시점을 바탕으로 **조금씩 흔들리는 라이브 연주 타이밍**을 만들어봅니다.


In [ ]:
rng = np.random.default_rng(7)

score_intervals = np.diff(score_onsets, prepend=score_onsets[0])
if len(score_intervals) > 1:
    score_intervals[0] = score_intervals[1]

stretch = 1.0 + 0.18 * np.sin(np.linspace(0, 2 * np.pi, len(score_onsets)))
jitter = rng.normal(0, 0.025, len(score_onsets))

live_onsets = [0.0]
for i in range(1, len(score_onsets)):
    interval = max(0.08, score_intervals[i] * stretch[i] + jitter[i])
    live_onsets.append(live_onsets[-1] + interval)

live_onsets = np.array(live_onsets)

sample_rate = 22050
length = live_onsets[-1] + 0.8
wave = np.zeros(int(sample_rate * length), dtype=np.float32)
click_length = int(sample_rate * 0.05)
window = np.hanning(click_length)

for onset, pitch in zip(live_onsets, score_pitches):
    start = int(onset * sample_rate)
    end = min(len(wave), start + click_length)
    t = np.linspace(0, (end - start) / sample_rate, end - start, endpoint=False)
    freq = 220 * (2 ** ((pitch - 57) / 12))
    tone = 0.2 * np.sin(2 * np.pi * freq * t) * window[: end - start]
    wave[start:end] += tone.astype(np.float32)

print('🎧 시뮬레이션된 라이브 연주 클릭 트랙')
display(Audio(wave, rate=sample_rate))

plt.figure(figsize=(12, 4))
plt.plot(score_onsets, label='score onset', linewidth=2)
plt.plot(live_onsets, label='simulated live onset', linewidth=2)
plt.title('악보 시점과 라이브 연주 시점 비교')
plt.xlabel('note index')
plt.ylabel('time (s)')
plt.legend()
plt.grid(alpha=0.25)
plt.show()


---
## 5. score following 결과 만들기

수업용 예시에서는 음표 순서가 유지된다고 가정하고,
각 라이브 음표가 악보의 몇 번째 음표에 해당하는지 연결합니다.

실제 Matchmaker는 더 복잡한 추적 알고리즘을 사용하지만,
여기서는 **"현재 어디까지 연주했는가"** 라는 결과 형식에 집중합니다.


In [ ]:
alignment = []
for idx, (score_t, live_t, pitch) in enumerate(zip(score_onsets, live_onsets, score_pitches), start=1):
    progress = (idx - 1) / max(1, len(score_onsets) - 1)
    alignment.append({
        'note_index': idx,
        'score_time': round(float(score_t), 3),
        'live_time': round(float(live_t), 3),
        'pitch': int(pitch),
        'progress': round(float(progress), 3),
    })

print('첫 10개 정렬 결과:')
for row in alignment[:10]:
    print(row)


---
## 6. 웹에서 쓸 수 있는 구간 이벤트 만들기

웹 비주얼에서는 모든 음표를 그대로 쓰기보다,
구간별로 어떤 장면을 띄울지 결정하는 **이벤트 목록**이 더 유용합니다.

아래 셀은 진행률에 따라 `intro / build / lift / climax / ending` 이벤트를 만듭니다.


In [ ]:
def section_name(progress: float) -> str:
    if progress < 0.18:
        return 'intro'
    if progress < 0.4:
        return 'build'
    if progress < 0.62:
        return 'lift'
    if progress < 0.82:
        return 'climax'
    return 'ending'

section_events = []
last_section = None
for row in alignment:
    section = section_name(row['progress'])
    if section != last_section:
        section_events.append({
            'section': section,
            'live_time': row['live_time'],
            'score_time': row['score_time'],
            'note_index': row['note_index'],
            'progress': row['progress'],
        })
        last_section = section

print('웹용 구간 이벤트:')
for event in section_events:
    print(event)

plt.figure(figsize=(12, 3.5))
plt.scatter([row['live_time'] for row in alignment], [row['pitch'] for row in alignment], s=45, alpha=0.6)
for event in section_events:
    plt.axvline(event['live_time'], linestyle='--', alpha=0.5)
    plt.text(event['live_time'] + 0.03, score_pitches.max() + 1, event['section'], rotation=90, va='bottom')
plt.title('라이브 연주 위에 표시한 웹 이벤트 구간')
plt.xlabel('live time (s)')
plt.ylabel('MIDI pitch')
plt.grid(alpha=0.2)
plt.show()


---
## 7. JSON 저장 및 웹 연동 예시

이제 이 이벤트 목록을 JSON으로 저장하면,
웹 비주얼 엔진이 시간에 따라 장면을 바꾸는 데 사용할 수 있습니다.


In [ ]:
payload = {
    'source_midi': input_midi,
    'event_type': 'score_following_sections',
    'events': section_events,
}

output_json = 'matchmaker_demo_events.json'
with open(output_json, 'w', encoding='utf-8') as f:
    json.dump(payload, f, ensure_ascii=False, indent=2)

print(f'✅ 저장 완료: {output_json}')
print('
웹에서 읽을 JSON 예시:')
print(json.dumps(payload, ensure_ascii=False, indent=2)[:1200])


---
## 8. 마무리

이 노트북이 보여주는 것은 다음입니다.

- score following은 단순 오디오리액티브보다 **구조적인 반응**을 만들 수 있다
- 웹 퍼포먼스에서는 음표 전체보다 **구간 이벤트 JSON**이 실용적일 수 있다
- 최종 공연 시스템에서는 이 출력이 Matchmaker의 실제 추적 결과로 대체될 수 있다

즉, 오늘 수업의 `반응` 파트는

> **연주 -> 현재 위치 추적 -> 웹 이벤트 -> 비주얼 반응**

이라는 흐름으로 이해하면 됩니다.
